# Practice Lab: Streamlining Preprocessing with Pipelines

In real-world data science, datasets are rarely clean. You will encounter a mix of numerical features with different scales, distributions, and sometimes missing values. Furthermore, **why** data needs different treatment matters. We cannot simply apply the same transformation to every column.

Today, you will learn the industry standard for handling messy data safely and efficiently: **Scikit-Learn's Pipelines** and **ColumnTransformers**.

### California Housing
We are using the California Housing dataset, which contains information about housing districts in California. It includes 8 numerical features describing each district, such as median income, house age, and room counts.

**The Business Problem:** A real estate analytics firm wants to predict the median house value (`MedHouseVal`) for California districts based on their demographic and geographic characteristics.

**Data Dictionary:**
| Column Name | Definition |
|-------------|------------|
|`MedInc`|: Median income in the block group (in $10k). *(Highly skewed — most districts have moderate income, some are very wealthy)*|
|`HouseAge`|: Median house age in the block group.|
|`AveRooms`|: Average number of rooms per household.|
|`AveBedrms`|: Average number of bedrooms per household.|
|`Population`|: Block group population.|
|`AveOccup`|: Average household occupancy (people per household).|
|`Latitude`|: Latitude of the block group.|
|`Longitude`|: Longitude of the block group.|
|`MedHouseVal`|: Median house value in the block group (in $100k). **Target variable.**|

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import sklearn
sklearn.set_config(transform_output="pandas") # Forces all transformers to output DataFrames!

# Data Loading & Splitting
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, OneHotEncoder, OrdinalEncoder

# Pipelines and Transformers
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Modeling & Evaluation
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

## 1. Load the Data
We will load the California Housing dataset using Scikit-Learn's built-in `fetch_california_housing()`. The data comes as a single `DataFrame` with 9 columns (8 features + 1 target).

In [2]:
# Load California Housing dataset (bundled with sklearn — no download needed)
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df = housing.frame  # Contains both features AND target column

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 2. Exploratory Data Analysis (EDA)

Before we can build our pipelines, we need to understand the shape of our data. Our preprocessing strategy depends entirely on the issues we uncover here.

**Our EDA Checklist:**
1. Check the first 5 rows to get a feel for the data.
2. Use `.info()` and `.isna().sum()` to identify data types and locate missing values.
3. Use `.describe()` for basic summary statistics.
4. Evaluate skewness to choose our mathematical transformations.
5. Look at the distribution of our numerical features.

__1. Display the first 5 rows and look at the data we will be working on__

__2.1 Get a general information on the data using `.info()`__

In [23]:
print(df.info)

<bound method DataFrame.info of        MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0      8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1      8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2      7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3      5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4      3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   
...       ...       ...       ...        ...         ...       ...       ...   
20635  1.5603      25.0  5.045455   1.133333       845.0  2.560606     39.48   
20636  2.5568      18.0  6.114035   1.315789       356.0  3.122807     39.49   
20637  1.7000      17.0  5.205543   1.120092      1007.0  2.325635     39.43   
20638  1.8672      18.0  5.329513   1.171920       741.0  2.123209     39.43   
20639  2.3886      16.0  5.254717   1.162264      1387.0  2.616981     39.37   

       

__2.2 Check for Missing Values__
>**Note on Missing Values:** In a standard workflow, when you see missing values during EDA, your instinct might be to fix them right away. **Don't!** We are just observing the mess right now. We will handle the actual filling (imputation) dynamically during the Data Preprocessing stage to prevent data leakage.

In [4]:
df.isna().sum()

MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

__Note on Missing Values:__ In a standard workflow, when you see missing values during EDA, your instinct might be to fix them right away. **Don't!** We are just observing the mess right now. We will handle the actual filling (imputation) dynamically during the Data Preprocessing stage to prevent data leakage.

In [5]:
# The California Housing dataset has no missing values in any column
df.isna().sum()

MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

__3. Look at the Summary statistics using `.describe()`__

In [6]:
df.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


>- Look closely at the summary statistics. Notice the range of values: `MedInc` goes from 0.5 to 15.0, while `AveOccup` ranges from 0.5 to 1243! These massive scale differences mean we **must** scale our features. If we used a model without scaling, `AveOccup` would dominate purely because of its large numbers.
>- `AveRooms` and `AveBedrms` have suspiciously high maximums (141 and 34, respectively). These are likely data errors or extreme outliers. StandardScaler will help reduce their impact compared to MinMaxScaler.

__4. Evaluate skewness to choose our mathematical transformations__

Some features like `MedInc`, `AveRooms`, `AveBedrms`, `Population`, and `AveOccup` are likely to be right-skewed. A `PowerTransformer` (Box-Cox or Yeo-Johnson) can make their distributions more normal, which helps linear models perform better.

In [7]:
# Import the skewness function to evaluate column distributions
from scipy.stats import skew

def check_skewness(df):
    """Return skewness value for each numeric column (excluding the target)."""
    X = df.drop('MedHouseVal', axis=1)
    return X.select_dtypes('number').apply(lambda col: skew(col.dropna())).sort_values(ascending=False)

check_skewness(df)

AveOccup      97.632465
AveBedrms     31.314680
AveRooms      20.696365
Population     4.935500
MedInc         1.646537
Latitude       0.465919
HouseAge       0.060326
Longitude     -0.297780
dtype: float64

In [8]:
# Run the function to check skewness
check_skewness(df)

AveOccup      97.632465
AveBedrms     31.314680
AveRooms      20.696365
Population     4.935500
MedInc         1.646537
Latitude       0.465919
HouseAge       0.060326
Longitude     -0.297780
dtype: float64

__5. Check the feature distributions__

Let's look at the spread of our features. `MedInc` (median income) and `AveOccup` (avg occupancy) have very different scales — a reminder of why scaling is essential before feeding data into a model.

In [9]:
# Check feature ranges
df.describe().loc[['min', 'max', 'mean', 'std']]

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956


__Summary of our preprocessing plan:__

Based on the EDA, here is our strategy:
- **Skewed features** (`MedInc`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`): Apply `PowerTransformer` to normalize distributions, then `StandardScaler`.
- **Symmetric features** (`HouseAge`, `Latitude`, `Longitude`): Apply `StandardScaler` directly.

In [10]:
# All our features are numeric — no columns to drop for this exercise
print("Features in the dataset:", list(df.columns))

Features in the dataset: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']


In [11]:
# Let's verify there are no missing values to handle
print(f"Total missing values in dataset: {df.isna().sum().sum()}")

Total missing values in dataset: 0


__Our target variable__

`MedHouseVal` is the median house value for each district (in units of $100,000). Our goal is to predict this value based on the 8 features we explored above.

In [12]:
# Confirm the feature names we'll use in our ColumnTransformer
feature_names = [col for col in df.columns if col != 'MedHouseVal']
print(f"Features ({len(feature_names)}): {feature_names}")
print(f"Target: MedHouseVal")

Features (8): ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Target: MedHouseVal


## 3. Train-Test Split

Before we do **any** preprocessing (like scaling numbers), we must split our data into training and testing sets.

**Why? To prevent Data Leakage!**
If we calculate scaling factors using the entire dataset, our training process would secretly learn information about the test set. By splitting first, our pipelines will be forced to calculate means and standard deviations using *only* the training data. The test data remains completely unseen, simulating how a model operates in the real world.

In [24]:
# Separate features (X) and target (y)
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

# Perform train-test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify the split
print(f"Training set: {X_train.shape}  |  Testing set: {X_test.shape}")

Training set: (16512, 8)  |  Testing set: (4128, 8)


## 4. The Hero - ColumnTransformer

To apply different preprocessing to different columns, we use a **`ColumnTransformer`**. 

If a Pipeline is a single factory assembly line, a ColumnTransformer is the factory manager. It allows you to apply different pipelines to different subsets of your data, and then seamlessly stitches the arrays back together at the end.

Think of it as a traffic cop routing data based on our EDA findings:

| Branch | Features | Preprocessing |
|--------|----------|--------------|
| **Branch 1 (Skewed)** | `MedInc`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup` | → PowerTransformer → StandardScaler |
| **Branch 2 (Symmetric)** | `HouseAge`, `Latitude`, `Longitude` | → StandardScaler |

In [14]:
# First, Define the columns for each branch
skewed_cols = ['MedInc', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup']  # Branch 1: skewed → PowerTransform → Scale
symmetric_cols = ['HouseAge', 'Latitude', 'Longitude']                       # Branch 2: symmetric → Scale only

In [15]:
## ==> BRANCH 1 (Skewed Numerics): Impute Median → PowerTransform → StandardScaler
skewed_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # Safety net — even though CA Housing has no NaNs
    ('power', PowerTransformer()),                    # Normalize skewed distributions
    ('scaler', StandardScaler())                      # Standardize to mean=0, std=1
])

In [16]:
## ==> BRANCH 2 (Symmetric Numerics): Impute Median → StandardScaler
symmetric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # Safety net
    ('scaler', StandardScaler())
])

In [17]:
# Finally, Combine all branches into the ColumnTransformer
preprocessor = ColumnTransformer([
    ('skewed_branch', skewed_pipeline, skewed_cols),
    ('symmetric_branch', symmetric_pipeline, symmetric_cols)
])

In [18]:
# (ColumnTransformer is already built above — all branches combined)

In [19]:
# Note: For the California Housing dataset, all features are numeric.
# If you had text columns, you would add OneHotEncoder/OrdinalEncoder branches here.

In [20]:
# preprocessor is ready above — let's move to the Master Pipeline!

## 5. The Master Pipeline & Evaluation

Now we bundle our `preprocessor` and our `LinearRegression` model into one final **Master Pipeline**. 

Think about all the work we did above. By using this pipeline, we can apply all preprocessing steps — median imputation, power transformations, and scaling — to our raw data with **a single call to `.fit()`**.

In [21]:
# Helper function to evaluate our models (calculates R-squared and RMSE for train and test sets)
def evaluate_model(model, X_train, y_train, X_test, y_test):
    """Print R² and RMSE for both train and test sets."""
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    print(f"Train R²:  {r2_train:.4f}")
    print(f"Test R²:   {r2_test:.4f}")
    print(f"Train RMSE: ${rmse_train:,.2f}")
    print(f"Test RMSE:  ${rmse_test:,.2f}")

In [22]:
# 1. Create the final Main Pipeline
main_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# 2. Fit the main_pipeline on the RAW training data
# We don't have to use our manually processed data — the pipeline does it all!
main_pipeline.fit(X_train, y_train)

# 3. Evaluate the main_pipeline
evaluate_model(main_pipeline, X_train, y_train, X_test, y_test)

Train R²:  0.6297
Test R²:   0.6133
Train RMSE: $0.70
Test RMSE:  $0.71


## 6. Reflection

**Your Task:**
Double-click this text cell. In 2-3 sentences, explain why passing new, unseen housing data through a `Pipeline` is significantly safer for production deployment than the manual approach.

> **SOLUTION:** A Pipeline bundles every preprocessing step (imputation, transformation, scaling) together with the model into a single object. When new data comes in, one call to `.predict()` applies the *exact same* transformations that were learned during training — eliminating the risk of forgetting a step or applying inconsistent logic. Manual preprocessing requires the data scientist to re-run every transformation by hand, which is error-prone and hard to maintain in production.

## 7. Communicating Results to Stakeholders

**How to phrase your answer:**
When speaking to stakeholders, avoid throwing raw math at them. Translate the metrics into real-world impact.

*Example using a Real Estate Model:*
*   ❌ **Bad:** "The model has an RMSE of 0.65 and an $R^2$ of 0.62."
*   ✅ **Good:** "Our model explains about 62% of the variation in median house values across California districts. When it predicts a district's median home value, it is typically off by about $65,000 on average."

**Your Task:**
Double-click this text cell. Based on your best model's RMSE and $R^2$ scores, write a 3-4 sentence explanation.
> **SOLUTION:**"Our real estate pricing model successfully accounts for about 61% of the factors that drive median house values across different California districts. When estimating the median home value for a new, unseen neighborhood, our predictions are typically off by about $71,000 on average. While this provides a strong and reliable baseline for understanding broader market trends, we may want to incorporate additional local data to tighten our estimates before making high-stakes investment decisions."